# AutoMode NeurIPS — Driver for GPU 0 (RTX PRO 6000, 96 GB)

**Role**: this GPU runs the big models — Gemma-2-9B and LLaMA-3-8B.

**What it does**:
- Builds the tiered experiment grid (Tier 0 headline, Tier 1 generality, Tier 3 ablations, Tier 4 supplementary)
- Filters to only the runs assigned to this GPU via `DEFAULT_GPU_ASSIGNMENT`
- Executes them sequentially with resume safety

**Prerequisites**:
1. `cd /path/to/automode_pkg && pip install -e .`
2. Export `HF_TOKEN` with access to gated Gemma/LLaMA checkpoints.
3. Run `pytest tests/test_switching.py tests/test_training_invariants.py -v` and confirm ALL tests pass before kicking off long runs.

In [ ]:
# ── One-time setup ────────────────────────────────────────────────────────
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'              # this shell sees only the 6000
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
# export HF_TOKEN in the shell before launching jupyter, or set it here:
# os.environ['HF_TOKEN'] = 'hf_...'

import torch
print(f'torch {torch.__version__} | cuda={torch.cuda.is_available()} | '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

In [ ]:
# ── Run unit tests FIRST — this is the safety net ─────────────────────────
# The test_promote_increases_trainable_count test is the one that would
# have caught the prefix bug in the original 2B notebook. If it fails,
# DO NOT proceed to any training runs.

import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_switching.py', 'tests/test_training_invariants.py', '-v'],
    cwd='..', capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, 'Unit tests FAILED. Do not proceed to training.'

In [ ]:
# ── Build the grid and filter to THIS GPU's slice ────────────────────────
from automode.grid import (
    build_tier_grid,
    shard_grid_by_gpu,
    DEFAULT_GPU_ASSIGNMENT,
)

GPU_RANK = 0  # RTX PRO 6000 is rank 0 in DEFAULT_GPU_ASSIGNMENT

# Tier order: 0 (headline) first, then 1 (generality), then 4 (supplementary).
# Tier 3 (ablations) is all 2B on GPU rank 2, so this GPU skips it.
full_grid = build_tier_grid(tiers=[0, 1, 4], output_root='runs')

mine = shard_grid_by_gpu(
    full_grid, gpu_rank=GPU_RANK, assignment=DEFAULT_GPU_ASSIGNMENT,
)

# Override device pinning for all my runs
for c in mine:
    c.device = 'cuda:0'

print(f'Total grid: {len(full_grid)} runs')
print(f'My slice (GPU {GPU_RANK}): {len(mine)} runs')
print()
# Show the first few so we can eyeball the plan
from collections import Counter
by_model = Counter(c.model_name for c in mine)
by_method = Counter(c.variant_label() for c in mine)
by_track = Counter(c.train_track for c in mine)
by_tier = Counter(c.tier for c in mine)
print(f'By model:  {dict(by_model)}')
print(f'By track:  {dict(by_track)}')
print(f'By tier:   {dict(by_tier)}')
print(f'By method (top 8): {dict(by_method.most_common(8))}')

In [ ]:
# ── Smoke test: one fast config end-to-end before kicking off the full grid ──
# This catches data-loading / tokenizer / model-loading errors early.
# Uses max_train_samples=32 so it finishes in ~5 min on a 9B.

from copy import deepcopy
from automode.config import preset_automode
from automode.train import run_experiment
from automode.grid import _apply_model_defaults, _apply_track_defaults

smoke = preset_automode(
    u=10, t=10, r=16, alpha=32,
    model_name='google/gemma-2-9b',
    train_track='gsm8k',
    eval_benchmarks=('gsm8k',),
    seed=42, epochs=1,
    output_root='runs/smoke',
    max_train_samples=32, max_eval_samples=50,
    tier=-1,
)
smoke = _apply_track_defaults(_apply_model_defaults(smoke))
smoke.device = 'cuda:0'

smoke_result = run_experiment(smoke)
assert smoke_result['status'] == 'ok', f'Smoke test failed: {smoke_result}'
print('\nSmoke test passed:')
print({k: v for k, v in smoke_result.items() if k != 'config'})

In [ ]:
# ── Kick off the real grid. Resume-safe — if the process dies, re-run this cell. ──
from automode.grid import run_grid
import time

CSV_PATH = 'runs/gpu0_results.csv'

t0 = time.time()
results = run_grid(
    mine,
    csv_path=CSV_PATH,
    stop_on_error=False,
    verbose=True,
)
elapsed_hr = (time.time() - t0) / 3600
print(f'\nGPU 0 complete: {len(results)} rows in {elapsed_hr:.1f} hr')
print(f'CSV written to: {CSV_PATH}')

In [ ]:
# ── Quick summary of what finished ────────────────────────────────────────
import pandas as pd
df = pd.read_csv(CSV_PATH)
print(f'{len(df)} runs completed')
print('\nBy model × method × track (mean eval accuracy):')
metric_cols = [c for c in df.columns if c.startswith('eval_') and ('maj1' in c or '_acc' in c)]
if metric_cols:
    summary = (df.groupby(['model_name', 'variant', 'train_track'])[metric_cols]
                 .mean()
                 .round(4))
    print(summary)